# different ways of creating a tool in langchain

In [10]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
import os

llm = init_chat_model(model="gpt-4.1-mini")
print(llm)
print(llm.invoke("hello"))

output_version=None profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x121072410> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12107b3d0> root_client=<openai.OpenAI object at 0x120d273d0> root_async_client=<openai.AsyncOpenAI object at 0x12107bfd0> model_name='gpt-4.1-mini' model_kwargs={} openai_api_key=SecretStr('**********') opena

### Tool creation through annotation

In [16]:
from langchain.tools import tool

@tool
def multiply_tool(x: int, y: int) -> int:
    """
    Multiply two numbers

    Args:
        x: The first number
        y: The second number

    Returns:
        The product of x and y
    """
    return x * y

print(multiply_tool)


name='multiply_tool' description='Multiply two numbers\n\nArgs:\n    x: The first number\n    y: The second number\n\nReturns:\n    The product of x and y' args_schema=<class 'langchain_core.utils.pydantic.multiply_tool'> func=<function multiply_tool at 0x1215fc2c0>


### Tool creation through  `StructuredTool`

In [24]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    location: str

def get_weather_tool(location: str):
    """
    Returns the weather in a given location

    Args:
        location: The location to get the weather for

    Returns:
        The weather in the given location
    """
    return f"The current weather in {location} is sunny"

weather_tool = StructuredTool.from_function(
    func=get_weather_tool,
    name="get_weather",
    description="Returns the weather in a given location",
    args_schema=WeatherInput,
)

print(weather_tool)


name='get_weather' description='Returns the weather in a given location' args_schema=<class '__main__.WeatherInput'> func=<function get_weather_tool at 0x12163e980>


### From Inheriting `StructuredTool` class into the custom tool class

In [26]:
from typing import ClassVar

class CalculatorInput(BaseModel):
    num1: int = Field(description="The first number")
    num2: int = Field(description="The second number")

class CalculatorTool(StructuredTool):
    name: ClassVar[str] = "Calculator"
    description: ClassVar[str] = "A tool for performing basic arithmetic operations when given two numbers"
    args_schema: ClassVar[type[BaseModel]] = CalculatorInput

    def _run(self, num1, num2):
        return num1 + num2

In [27]:
calculator = CalculatorTool()
tools = [multiply_tool, weather_tool, calculator]
llm_with_tools = llm.bind_tools(tools)
print(llm_with_tools)

bound=ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x121072410>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12107b3d0>, root_client=<openai.OpenAI object at 0x120d273d0>, root_async_client=<openai.AsyncOpenAI object at 0x12107bfd0>, model_name='gpt-4.1-mini', model_kwargs={}, openai_api_key=Secr

### Invoke LLM with each tool

In [28]:
# multiply_tool
response = llm_with_tools.invoke("What is 6 multiplied by 7?")
print(response.tool_calls)

[{'name': 'multiply_tool', 'args': {'x': 6, 'y': 7}, 'id': 'call_N7uWQviTAJbhhAKTgyMABtQC', 'type': 'tool_call'}]


In [29]:
# weather_tool
response = llm_with_tools.invoke("What is the weather in Paris?")
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'Paris'}, 'id': 'call_AMj51giMUotK3G2Eg7CLCQfd', 'type': 'tool_call'}]


In [30]:
# CalculatorTool
response = llm_with_tools.invoke("What is 15 plus 27?")
print(response.tool_calls)

[{'name': 'Calculator', 'args': {'num1': 15, 'num2': 27}, 'id': 'call_b939kTpJH0EjquhGfCd7ASnl', 'type': 'tool_call'}]
